In [3]:
import polars as pl
import json

# Load CSV
df = pl.read_csv("/home/mahdi/Loan_Reommender_System/wizard_solution/MEC-LoanRecomn_Scenarios-V0.7.csv", null_values=["nan"])

# Convert to list of dictionaries (row-oriented)
loans = df.to_dicts()
print(len(loans))

# Dump to JSON string
json_str = json.dumps(loans, indent=4)

# Save to file
with open("output.json", "w") as list_loans:
    list_loans.write(json_str)



211


In [ ]:
def filter_loans(loans, deposit_amount=None, deposit_duration=None, 
                 loan_amount=None, credit_score=None, interest_rate=None, 
                 repayment_duration=None):
    
    # if loan_amount or deposit_amount is not None:
    #     if deposit_duration is None:

    #         deposit_duration= [2,3]

    #     if credit_score is None:

    #         credit_score = "B"            



    params = {
        "deposit_amount": deposit_amount,
        "deposit_duration": deposit_duration,
        "loan_amount": loan_amount,  
        "credit_score": credit_score,
        "interest_rate": interest_rate,
        "repayment_duration": repayment_duration,
    }
    
    filtered_loans = [params] 
    
    
    for loan in loans:
        conditions = []  
        for key, value in params.items():
            
            if value is not None:
                if key == "loan_amount":
                    
                    conditions.append(loan.get("loan_amount_limit", 0) >= value)
                # elif key == "deposit_amount":

                #     conditions.append((loan.get("maximum_deposit_amount", 0) >= value) )
                # elif key == "deposit_duration":
                #     # If deposit_duration is a list, check membership.
                #     if isinstance(value, list):
                #         conditions.append(loan.get(key) in value)
                #     else:
                #         conditions.append(loan.get(key) == value)    
                
                else:
                    conditions.append(loan.get(key) == value)
        # If at least one condition was provided and all are True, include the loan.
        if conditions and all(conditions):
            filtered_loans.append(loan)
    
    return filtered_loans


In [22]:
def filter_loans(loans, deposit_amount=None, deposit_duration=None, 
                 loan_amount=None, credit_score=None, interest_rate=None, 
                 repayment_duration=None):
    
    params = {
        "deposit_amount": deposit_amount,
        "deposit_duration": deposit_duration,
        "loan_amount": loan_amount,  
        "credit_score": credit_score,
        "interest_rate": interest_rate,
        "repayment_duration": repayment_duration,
    }

    filtered_loans = []

    for loan in loans:
        loan_copy = loan.copy()  # Avoid mutating the original loan data
        
        # Step 1: Apply the loan_amount logic
        if loan_amount is not None:
            loan_copy["loan_amount"] = loan_amount
            
            # Check that loan_coefficient exists and is not None or zero
            coefficient = loan_copy.get('loan_coefficient')
            if coefficient:
                loan_copy["deposit_amount"] = loan_amount / coefficient
            else:
                loan_copy["deposit_amount"] = None  # Or set to 0 or skip, depending on your logic
        # Step 2: Check conditions
        conditions = []  
        for key, value in params.items():
            if value is not None:
                if key == "loan_amount":
                    # Check if loan's limit is enough
                    conditions.append(loan.get("loan_amount_limit", 0) >= value)
                else:
                    conditions.append(loan_copy.get(key) == value)

        if conditions and all(conditions):
            filtered_loans.append(loan_copy)
    
    return filtered_loans


In [25]:
filtered = filter_loans(loans, loan_amount= 1000000000, credit_score= "B")
print(len(filtered))
print("Filtered loans:")
print(filtered)
# for loan in filtered:
#     print(loan)

72
Filtered loans:
[{'alias': 'شایان', 'package_name': 'شایان یک', 'contract_type': 'مرابحه', 'granted_method': 'واریز به حساب', 'loan_amount_limit': 3000000000, 'deposit_duration': 1, 'interest_rate': 23, 'repayment_duration': 12, 'loan_coefficient': 50, 'credit_score': 'B', 'minimum_deposit_amount': None, 'maximum_deposit_amount': 15000000000, 'minimum_loan_amount': 100000000, 'guarantee': 1, 'receiving_channel': 'غیر حضوری/ حضوری', 'account_type': None, 'loan_amount': 1000000000, 'deposit_amount': 20000000.0}, {'alias': 'شایان', 'package_name': 'شایان یک', 'contract_type': 'جعاله', 'granted_method': 'واریز به حساب', 'loan_amount_limit': 3000000000, 'deposit_duration': 1, 'interest_rate': 23, 'repayment_duration': 12, 'loan_coefficient': 50, 'credit_score': 'B', 'minimum_deposit_amount': None, 'maximum_deposit_amount': 15000000000, 'minimum_loan_amount': 100000000, 'guarantee': 1, 'receiving_channel': 'غیر حضوری/ حضوری', 'account_type': None, 'loan_amount': 1000000000, 'deposit_amoun

In [ ]:
def filter_loans(loans, deposit_amount=None, deposit_duration=None, 
                 loan_amount=None, credit_score=None, interest_rate=None, 
                 repayment_duration=None):
    
    params = {
        "deposit_amount": deposit_amount,
        "deposit_duration": deposit_duration,
        "loan_amount": loan_amount,  
        "credit_score": credit_score,
        "interest_rate": interest_rate,
        "repayment_duration": repayment_duration,
    }

    updated_loans = []

    # Step 1: Update all loans with loan_amount logic
    for loan in loans:
        loan_copy = loan.copy()

        if loan_amount is not None:
            loan_copy["loan_amount"] = loan_amount
            
            # Placeholder formulas for demonstration
            loan_copy["deposit_amount"] = loan_amount * 0.1  # 10% of loan amount
            loan_copy["repayment_amount"] = loan_amount * 1.05  # 5% interest

        updated_loans.append(loan_copy)

    # Step 2: Filter based on updated data
    filtered_loans = []
    for loan in updated_loans:
        conditions = []
        for key, value in params.items():
            if value is not None:
                if key == "loan_amount":
                    # Check against the limit only, not the updated value
                    conditions.append(loan.get("loan_amount_limit", 0) >= value)
                else:
                    conditions.append(loan.get(key) == value)

        if conditions and all(conditions):
            filtered_loans.append(loan)

    return filtered_loans
